### Bronze Ingestion – Menu Items (JSON)

This notebook ingests menu_items JSON files from Unity Catalog Volumes
into the Bronze layer using Databricks Auto Loader.The ingestion is idempotent and adds audit and lineage metadata for governance and traceability.


In [0]:
dbutils.widgets.text("catalog", "coffee")
dbutils.widgets.text("bronze_schema", "bronze")
dbutils.widgets.text("table_name", "menu_items")
dbutils.widgets.text("raw_volume", "/Volumes/workspace/default/coffee_raw_volume")

catalog = dbutils.widgets.get("catalog")
bronze_schema = dbutils.widgets.get("bronze_schema")
table_name = dbutils.widgets.get("table_name")
raw_volume = dbutils.widgets.get("raw_volume")

# -------------------------
# Build dynamic paths
# -------------------------
source_path = f"{raw_volume}/menu_items_json/"
checkpoint_base = f"/Volumes/workspace/default/coffee_raw_volume/_checkpoints/bronze/menu_items"
target_table = f"{catalog}.{bronze_schema}.{table_name}"

In [0]:
# format("cloudFiles") enables Auto Loader
# cloudFiles.format = json tells Auto Loader the file type
# schemaLocation stores inferred schema
# inferColumnTypes = false keeps everything as STRING in Bronze

from pyspark.sql.functions import current_timestamp, current_date, input_file_name
from pyspark.sql.types import StructType, StructField, StringType

menu_schema = StructType([
    StructField("item_id", StringType(), True),
    StructField("item_name", StringType(), True),
    StructField("category", StringType(), True),
    StructField("price", StringType(), True),
    StructField("is_seasonal", StringType(), True),
    StructField("available_from", StringType(), True),
    StructField("available_to", StringType(), True)
])

# -------------------------
# Read Stream (Auto Loader)
# -------------------------
df = (
    spark.readStream
      .format("cloudFiles")
      .option("cloudFiles.format", "json")
      .schema(menu_schema)  # forces missing-null columns to exist
      .option("cloudFiles.schemaLocation", f"{checkpoint_base}/schema")
      .load(source_path)
)


In [0]:
from pyspark.sql.functions import current_timestamp, current_date, col

df_enriched = (
    df
    .withColumn("loaded_at", current_timestamp())
    .withColumn("updated_at", current_timestamp())
    .withColumn("load_dt", current_date())
    .withColumn("source_file", col("_metadata.file_path"))
)


In [0]:
# outputMode("append") → only new rows are appended
# trigger(availableNow=True) makes this behave like batch:
# - It processes all new files once
# - Then stops automatically
(
    df_enriched.writeStream
    .format("delta")
    .outputMode("append")
    .trigger(availableNow=True)
    .option("checkpointLocation", checkpoint_base + "/write")
    .toTable(target_table)
)
